In [ ]:
conda activate wes

In [ ]:
mkdir 00_ref # 参考数据文件夹
mkdir -p 01_fastp/report # 质控及质控报告
mkdir 02_bwa_out # 比对
mkdir 03_markdup # 标记重复
mkdir 04_bqsr # 重新校准碱基质量值
mkdir 05_gvcf # gvcf文件
mkdir 06_joint_genotype # joint-genotype
mkdir 07_VQSR # 变异质控
mkdir 08_annovar # 位点注释
mkdir tmp
cd 00_ref/
mkdir bed
mkdir gatk_call_vcf
mkdir hg38

# MD5 CHECK

In [ ]:
#shell check md5
cd /localdisk/immune/0_raw_immune1k/immune1K_WGS/WGS

my_func() {
  cd $1
  md5sum -c MD5.txt
}
export -f my_func
ls | parallel -j 12 my_func

# Get sample ID

In [ ]:
cd 01_fastp
path=/localdisk/immune/0_raw_immune1k/immune1K_WGS/WGS_fq_before_merge/*
for filename in $path
do
   echo $filename >> merge.list
done

# Merge

In [ ]:
#run
cd /home/liyanguo/MyImmuCell_WGS
mergefq_func() {
    local path=$1
    sample=$(basename $path)
    mkdir /localdisk/immune/0_raw_immune1k/immune1K_WGS/WGS/$sample
    cat $path/*_1.fq.gz > /localdisk/immune/0_raw_immune1k/immune1K_WGS/WGS/$sample/$sample'_1.fq.gz'
    cat $path/*_2.fq.gz > /localdisk/immune/0_raw_immune1k/immune1K_WGS/WGS/$sample/$sample'_2.fq.gz'
}
export -f mergefq_func
nohup cat 01_fastp/merge.list | nohup parallel -j 4 mergefq_func &

In [ ]:
#run generate md5
cd /home/liyanguo/MyImmuCell_WGS
md5sum_func() {
    local path=$1
    cd $path
    md5sum *_1.fq.gz > MD5.txt
    md5sum *_2.fq.gz >> MD5.txt
}
export -f md5sum_func
nohup cat a.txt | nohup parallel -j 2 md5sum_func &

# fastp

In [ ]:
#run
cd /home/liyanguo/MyImmuCell_WGS
fastp_func() {
    local path=$1
    sample=$(basename $path)
    fastp \
    -i $path/*_1.fq.gz -o /localdisk/immune/0_raw_immune1k/immune1K_WGS/WGS_analysis/$sample'_1.clean.fq.gz' \
    -I $path/*_2.fq.gz -O /localdisk/immune/0_raw_immune1k/immune1K_WGS/WGS_analysis/$sample'_2.clean.fq.gz' \
    -z 4 -f 5 -t 5 -F 5 -T 5 -5 -W 5 -M 20 -Q -l 50 -c -w 8 \
    -j 01_fastp/report/$sample.fastp.json \
    -h 01_fastp/report/$sample.fastp.html
}
export -f fastp_func
nohup cat 01_fastp/aa.txt | nohup parallel -j 1 fastp_func &

# bwa algin

## bwa index

In [ ]:
bwa index Homo_sapiens.GRCh38.dna.primary_assembly.fa
samtools faidx Homo_sapiens.GRCh38.dna.primary_assembly.fa # 创建fasta序列格式索引
gatk CreateSequenceDictionary -R Homo_sapiens.GRCh38.dna.primary_assembly.fa -O Homo_sapiens.GRCh38.dna.primary_assembly.dict # 生成参考基因组的dict文件

In [ ]:
cd 02_bwa_out
ls /localdisk/immune/0_raw_immune1k/immune1K_WGS/WGS/ > sample_id.list

In [ ]:
#run
cd /home/liyanguo/MyImmuCell_WGS
bwa_func() {
    local path=/localdisk/immune/0_raw_immune1k/immune1K_WGS/WGS_analysis
    local sample=$1
    bwa mem -t 12 -M -R '@RG\tID:onelane\tPL:UNKNOWN\tSM:'${sample} \
    00_ref/hg38/Homo_sapiens.GRCh38.dna.primary_assembly.fa \
    $path/$sample'_1.clean.fq.gz' \
    $path/$sample'_2.clean.fq.gz' \
    | samtools sort -@ 12 -O bam -o $path/$sample'_align_sorted.bam' -
}
export -f bwa_func
nohup parallel -j 1 -a 02_bwa_out/xaa bwa_func &

In [ ]:
while read -r i; do
    if ls /storage1/immune/0_raw_immune1k/immune1K_WGS/WGS_analysis/${i}_align_sorted_markdup_BQSR.bam &> /dev/null; then
        echo "$i 存在。"
    else
        echo "$i 不存在。"
    fi
done < 03_mark_dup/all_sample_id.list

# mark duplication

In [ ]:
#run
mark_duplication() {
    local path=/storage1/immune/0_raw_immune1k/immune1K_WGS/WGS_analysis
    local sample=$1
    gatk MarkDuplicates --java-options "-Xmx100G -Djava.io.tmpdir=./tmp" \
    -I $path/$sample'_align_sorted.bam' \
    -O $path/$sample'_align_sorted_markdup.bam' \
    -M $path/$sample'_markdup_matrics.txt'
    samtools index $path/$sample'_align_sorted_markdup.bam'
}
export -f mark_duplication

cat 03_mark_dup/all_sample_id.list | nohup parallel -j 6 mark_duplication > task.log 2>&1 &

# BaseRecalibrator

In [ ]:
#run
BaseRecalibrator() {
    local path=/storage1/immune/0_raw_immune1k/immune1K_WGS/WGS_analysis/
    sample=$1
    hg38_vcf=00_ref/gatk_call_vcf
    hg38_ref=00_ref/hg38/Homo_sapiens.GRCh38.dna.primary_assembly.fa
    gatk --java-options "-Xmx100G -Djava.io.tmpdir=./tmp" BaseRecalibrator \
    -R $hg38_ref \
    -I $path/${sample}_align_sorted_markdup.bam \
    --known-sites $hg38_vcf/1000G_phase1.snps.high_confidence.hg38.vcf \
    --known-sites $hg38_vcf/Mills_and_1000G_gold_standard.indels.hg38.vcf \
    --known-sites $hg38_vcf/Homo_sapiens_assembly38.dbsnp138.vcf \
    --known-sites $hg38_vcf/Homo_sapiens_assembly38.known_indels.vcf \
    -O 04_bqsr/${sample}_recal_data.table
}
export -f BaseRecalibrator
nohup cat 03_mark_dup/all_sample_id.list1 | nohup parallel -j 1 BaseRecalibrator &

# ApplyBQSR

In [ ]:
#run
ApplyBQSR() {
    local path=/storage1/immune/0_raw_immune1k/immune1K_WGS/WGS_analysis/
    sample=$1
    hg38_ref=00_ref/hg38/Homo_sapiens.GRCh38.dna.primary_assembly.fa

    gatk --java-options "-Xmx100G -Djava.io.tmpdir=./tmp" ApplyBQSR \
    -R $hg38_ref \
    -I $path/${sample}_align_sorted_markdup.bam \
    -O ./${sample}_align_sorted_markdup_BQSR.bam \
    -bqsr 04_bqsr/${sample}_recal_data.table
    mv ./${sample}_align_sorted_markdup_BQSR.bam $path &
    mv ./${sample}_align_sorted_markdup_BQSR.bai $path &
}
export -f ApplyBQSR
nohup cat 03_mark_dup/all_sample_id.list1 | nohup parallel -j 24 ApplyBQSR > task_ApplyBQSR.log 2>&1 &

# Germline mutation

## HaplotypeCaller

In [ ]:
#run
HaplotypeCaller() {
    local path=/storage1/immune/0_raw_immune1k/immune1K_WGS/WGS_analysis/
    sample=$1
    hg38_vcf=00_ref/gatk_call_vcf
    hg38_ref=00_ref/hg38/Homo_sapiens.GRCh38.dna.primary_assembly.fa

    cp $path/${sample}_align_sorted_markdup_BQSR.bam .
    cp $path/${sample}_align_sorted_markdup_BQSR.bai .
        
    gatk --java-options "-Xmx100G -Djava.io.tmpdir=./tmp" HaplotypeCaller \
    -ERC GVCF \
    -R $hg38_ref \
    --native-pair-hmm-threads 12 \
    -G StandardAnnotation \
    -G AS_StandardAnnotation \
    -G StandardHCAnnotation \
    -I ${sample}_align_sorted_markdup_BQSR.bam \
    -D $hg38_vcf/Homo_sapiens_assembly38.dbsnp138.vcf \
    -O 05_gvcf/${sample}_HaplotypeCaller.g.vcf.gz

    rm ${sample}_align_sorted_markdup_BQSR.bam
    rm ${sample}_align_sorted_markdup_BQSR.bai
}
export -f HaplotypeCaller
nohup parallel -j 16 HaplotypeCaller :::: 03_mark_dup/all_sample_id.list &

## Combine GVCFs

In [ ]:
bed=00_ref/hg38/Homo_sapiens.GRCh38.dna.primary_assembly.fa.fai
cat $bed | awk -F'\t' '{print $1}' | uniq > 06_joint_genotype/chrom.list

In [ ]:
GenomicsDBImport() {
    hg38_ref=/home/liyanguo/MyImmuCell_WGS/00_ref/hg38/Homo_sapiens.GRCh38.dna.primary_assembly.fa
    chrom=$1
    #rm 06_joint_genotype/${chrom} -r
    gatk GenomicsDBImport \
      --genomicsdb-workspace-path 06_joint_genotype/${chrom} \
      --batch-size 20 \
      --reader-threads 12 \
      -R ${hg38_ref} \
      -L ${chrom} \
      --sample-name-map 06_joint_genotype/cohort.sample_map \
      --tmp-dir tmp/
}
export -f GenomicsDBImport
nohup parallel -j 8 GenomicsDBImport :::: 06_joint_genotype/chrom.list &

## joint genotyping

In [ ]:
GenotypeGVCFs(){
    hg38_ref=/home/liyanguo/MyImmuCell_WGS/00_ref/hg38/Homo_sapiens.GRCh38.dna.primary_assembly.fa
    chrom=$1
    gatk --java-options "-Xmx100G -Djava.io.tmpdir=./tmp" GenotypeGVCFs \
    -R $hg38_ref \
    -V gendb:///storage1/immune/MyImmuCell_WGS/06_joint_genotype/${chrom} \
    -L ${chrom} \
    -O /storage1/immune/MyImmuCell_WGS/06_joint_genotype/WGS_variants_${chrom}.vcf
}
export -f GenotypeGVCFs
nohup parallel -j 2 GenotypeGVCFs :::: 06_joint_genotype/chrom.list &

# Merge

In [ ]:
nohup gatk MergeVcfs \
$(for i in $(ls /storage1/immune/MyImmuCell_WGS/06_joint_genotype/WGS_variants_*.vcf);do echo "-I $i";done) \
-O /storage1/immune/MyImmuCell_WGS/06_joint_genotype/WGS_variants_cohort.vcf &

# VQSR

## SNP

In [ ]:
# SNPs VQSR
nohup gatk --java-options "-Xmx100g -Xms100g" VariantRecalibrator \
    --trust-all-polymorphic \
    -V /storage1/immune/MyImmuCell_WGS/06_joint_genotype/WGS_variants_cohort.vcf \
    --resource:hapmap,known=false,training=true,truth=true,prior=15 $hg38_vcf/hapmap_3.3.hg38.vcf \
    --resource:omini,known=false,training=true,truth=true,prior=12 $hg38_vcf/1000G_omni2.5.hg38.vcf \
    --resource:1000G,known=false,training=true,truth=false,prior=10 $hg38_vcf/1000G_phase1.snps.high_confidence.hg38.vcf \
    --resource:dbsnp,known=true,training=false,truth=false,prior=7 $hg38_vcf/Homo_sapiens_assembly38.dbsnp138.vcf \
    -an FS -an ReadPosRankSum -an MQRankSum -an QD -an SOR -an DP \
    -tranche 100.0 -tranche 99.95 -tranche 99.9 -tranche 99.8 -tranche 99.6 -tranche 99.5 -tranche 99.4 -tranche 99.3 -tranche 99.0 -tranche 98.0 -tranche 97.0 -tranche 90.0 \
    -mode SNP \
    --max-gaussians 6 \
    --output /storage1/immune/MyImmuCell_WGS/07_snp/WGS_variants_SNPs.recal \
    --tranches-file /storage1/immune/MyImmuCell_WGS/07_snp/WGS_variants_SNPs.tranches &

In [ ]:
nohup gatk --java-options "-Xmx100g -Xms100g" \
    ApplyVQSR \
    -V /storage1/immune/MyImmuCell_WGS/06_joint_genotype/WGS_variants_cohort.vcf \
    --recal-file /storage1/immune/MyImmuCell_WGS/07_snp/WGS_variants_SNPs.recal \
    --tranches-file /storage1/immune/MyImmuCell_WGS/07_snp/WGS_variants_SNPs.tranches \
    --truth-sensitivity-filter-level 99.5 \
    --create-output-variant-index true \
    -mode SNP \
    -O /storage1/immune/MyImmuCell_WGS/07_snp/WGS_variants_SNPs.recalibrated.vcf &

## Indel

In [ ]:
nohup gatk --java-options "-Xmx100g -Xms100g" VariantRecalibrator \
    -V /storage1/immune/MyImmuCell_WGS/06_joint_genotype/WGS_variants_cohort.vcf \
    --trust-all-polymorphic \
    -tranche 100.0 -tranche 99.95 -tranche 99.9 -tranche 99.5 -tranche 99.0 -tranche 97.0 -tranche 96.0 -tranche 95.0 -tranche 94.0 -tranche 93.5 -tranche 93.0 -tranche 92.0 -tranche 91.0 -tranche 90.0 \
    -an FS -an ReadPosRankSum -an MQRankSum -an QD -an SOR -an DP \
    -mode INDEL \
    --max-gaussians 4 \
    -resource:mills,known=false,training=true,truth=true,prior=12 $hg38_vcf/Mills_and_1000G_gold_standard.indels.hg38.vcf \
    -resource:axiomPoly,known=false,training=true,truth=false,prior=10 $hg38_vcf/Axiom_Exome_Plus.genotypes.all_populations.poly.hg38.vcf \
    -resource:dbsnp,known=true,training=false,truth=false,prior=2 $hg38_vcf/Homo_sapiens_assembly38.dbsnp138.vcf \
    -O /storage1/immune/MyImmuCell_WGS/08_indel/WGS_variants_INDELs.recal \
    --tranches-file /storage1/immune/MyImmuCell_WGS/08_indel/WGS_variants_INDELs.tranches &

In [ ]:
nohup gatk --java-options "-Xmx100g -Xms100g" \
    ApplyVQSR \
    -V /storage1/immune/MyImmuCell_WGS/06_joint_genotype/WGS_variants_cohort.vcf \
    --recal-file /storage1/immune/MyImmuCell_WGS/08_indel/WGS_variants_INDELs.recal \
    --tranches-file /storage1/immune/MyImmuCell_WGS/08_indel/WGS_variants_INDELs.tranches \
    --truth-sensitivity-filter-level 99.0 \
    --create-output-variant-index true \
    -mode INDEL \
    -O /storage1/immune/MyImmuCell_WGS/08_indel/WGS_variants_INDELs.recalibrated.vcf &

# PLINK2

In [ ]:
bgzip /storage1/immune/MyImmuCell_WGS/07_snp/WGS_variants_SNPs.recalibrated.vcf
/storage1/immune/MyImmuCell_WGS/07_snp/
bcftools index WGS_variants_SNPs.recalibrated.vcf.gz

bcftools view \
    -r chr1,chr2,chr3,chr4,chr5,chr6,chr7,chr8,chr9,chr10,chr11,chr12,chr13,chr14,chr15,chr16,chr17,chr18,chr19,chr20,chr21,chr22 \
    /storage1/immune/MyImmuCell_WGS/07_snp/WGS_variants_SNPs.recalibrated.vcf.gz > ./MyImmuCell_WGS/09_snp_plink/WGS_variants_SNPs.recalibrated.autosome.vcf.gz
echo "bcftools done!"

# 转换VCF为PLINK格式
plink --vcf ./MyImmuCell_WGS/07_snp/WGS_variants_SNPs.recalibrated.autosome.vcf.gz \
    --make-bed \
    --out ./MyImmuCell_WGS/09_snp_plink/cohort/
echo "Cohort done!"